In [ ]:
# !pip install moomoo-api
# !pip install matplotlib inline
# !pip install finnhub-python
# !pip install python-dotenv
# !pip install playwright
# !playwright install chromium

184.3 MiB [                    ] 0% 0.0s184.3 MiB [                    ] 0% 210.6s184.3 MiB [                    ] 0% 246.7s184.3 MiB [                    ] 0% 815.5s184.3 MiB [                    ] 0% 682.7s184.3 MiB [                    ] 0% 925.5s184.3 MiB [                    ] 0% 806.7s184.3 MiB [                    ] 0% 730.2s184.3 MiB [                    ] 0% 675.8s184.3 MiB [                    ] 0% 630.8s184.3 MiB [                    ] 0% 627.8s184.3 MiB [                    ] 0% 590.0s184.3 MiB [                    ] 0% 563.4s184.3 MiB [                    ] 0% 542.7s184.3 MiB [                    ] 0% 522.4s184.3 MiB [                    ] 0% 503.2s184.3 MiB [                    ] 0% 487.2s184.3 MiB [                    ] 0% 470.3s184.3 MiB [                    ] 0% 439.3s184.3 MiB [                    ] 0% 410.3s184.3 MiB [                    ] 0% 384.8s184.3 MiB [                    ] 0% 362.9s184.3 MiB [                    ] 0% 344.7s184.3 MiB [                    ] 0% 

# 牛来

In [1]:
from moomoo import *
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import concurrent.futures
import urllib.request

# quote_ctx gets the market data from moomoo server, and trade_ctx gets the trading data from moomoo server.
quote_ctx = OpenQuoteContext(host='127.0.0.1', port=11111)  # Create quote object
print(quote_ctx.get_market_snapshot('HK.00700'))  # Get market snapshot for HK.00700
quote_ctx.close() # Close object to prevent the number of connextions from running out
print("----------------------------------------------")
trd_ctx = OpenSecTradeContext(host='127.0.0.1', port=11111)  # Create trade object
print(trd_ctx.place_order(price=500.0, qty=100, code="HK.00700", trd_side=TrdSide.BUY, trd_env=TrdEnv.SIMULATE))  # Placing an order through paper trading account (Live trading requires unlocking first; if using the GUI version of OpenD, skip this step and unlock manually by clicking the Unlock button on the UI)

trd_ctx.close()  # Close object to prevent the number of connextions from running out

2026-09-15 14:48:12,599 | 9900 | 135561859933696 | [open_context_base.py:411] _init_connect_sync: New connect ready: conn=7505517813841313403(1) context=<moomoo.quote.open_quote_context.OpenQuoteContext object at 0x7b4a9494c6e0>
(0,        code     name          update_time  last_price  open_price  high_price  \
0  HK.00700  TENCENT  2026-09-15 14:48:11       439.6       428.0       446.0   

   low_price  prev_close_price      volume      turnover  ...  \
0      428.0             430.6  16297322.0  7.165005e+09  ...   

   after_change_rate  after_amplitude overnight_price  overnight_high_price  \
0                N/A              N/A             N/A                   N/A   

   overnight_low_price  overnight_volume  overnight_turnover  \
0                  N/A               N/A                 N/A   

   overnight_change_val  overnight_change_rate  overnight_amplitude  
0                   N/A                    N/A                  N/A  

[1 rows x 142 columns])
--------------------

## Sentiment correlation in financial news networks and associated market movements pipeline

In [19]:
# load finnhub news
import os
from dotenv import load_dotenv
import finnhub
import datetime
import pandas as pd
import numpy as np
import requests
from bs4 import BeautifulSoup
import json
from playwright.async_api import async_playwright

load_dotenv(dotenv_path = '../main/.env')  
FINNHUB_API_KEY = os.getenv('FINNHUB_API_KEY')
# today_date = datetime.datetime.now().strftime("%Y-%m-%d")

finnhub_client = finnhub.Client(api_key=FINNHUB_API_KEY)

def date_days_ago(days, curr_date):
    return (curr_date - datetime.timedelta(days=days)).strftime("%Y-%m-%d")

def get_sixtyd_rolling_window(company_code):
    # uese yesterday date to avoid lookahead bias
    yesterday_date = (datetime.datetime.now() - datetime.timedelta(days=1))
    yesterday_date_str = yesterday_date.strftime("%Y-%m-%d")
    sixtyd_ago_date = date_days_ago(60, yesterday_date)
    sixtyd_window_news = finnhub_client.company_news(company_code, _from=sixtyd_ago_date, to=yesterday_date_str)
    return sixtyd_window_news


sixtyd_window_news_amzn = get_sixtyd_rolling_window('AMZN')

print(len(sixtyd_window_news_amzn))

sixtyd_window_news_amzn[0:5]

248


[{'category': 'company',
  'datetime': 1789429800,
  'headline': 'Finding Value Opportunities Among Highly Rated Stocks',
  'id': 142148394,
  'image': 'https://static.seekingalpha.com/cdn/s3/uploads/getty_images/2264414168/image_2264414168.jpg?io=getty-c-w1536',
  'related': 'AMZN',
  'source': 'SeekingAlpha',
  'summary': 'U.S. equity markets moved lower this week, with broad weakness across most major ETFs and sector benchmarks.',
  'url': 'https://finnhub.io/api/news?id=265f2051e062e7edaf4f0fe9ba5f0ed39af4a7efd11b0a552f57b04aee659c15'},
 {'category': 'company',
  'datetime': 1789427968,
  'headline': "Why QUALCOMM (QCOM) Is Up 6.8% After New Amazon AI Chip Deal And What's Next",
  'id': 142148398,
  'image': 'https://s.yimg.com/rz/stage/p/yahoo_finance_en-US_h_p_finance_2.png',
  'related': 'AMZN',
  'source': 'Yahoo',
  'summary': 'Earlier this month, Qualcomm Technologies announced a multi-generation collaboration with Amazon to develop customized AI data center silicon and high-

In [3]:
# 
us_company_codes = [
    # cloud
    "AAPL", "MSFT", "GOOGL", "AMZN", "META", "NFLX", "CRM", "ORCL",
    # chips
    "NVDA", "AMD", "AVGO", "INTC", "CSCO", "IBM",
    # retail
    "TSLA", "WMT", "TGT", "COST", "MCD", "SBUX", "GM", "HD",
    # banking
    "JPM", "GS", "MS", "BAC", "WFC", "AXP", "V", "MA", "BRK.B",
    # sector anchor
    "PG", "KO", "PEP", "JNJ", "PFE", "LLY", "UNH", "GE", "CAT", "BA", "XOM", "CVX"

]

news_dict = {}

for code in us_company_codes:
    try:
        news = get_sixtyd_rolling_window(code)
        news_dict[code] = news
        print(f"company {code} processed! Total of {len(news)} news")
    except Exception as e:
        print('error processing: '+ code + f"\n{e}")

# write to json
with open('news_dict.json', 'w') as f:
    json.dump(news_dict, f)

news_dict

company AAPL processed! Total of 243 news
company MSFT processed! Total of 249 news
company GOOGL processed! Total of 247 news
company AMZN processed! Total of 247 news
company META processed! Total of 249 news
company NFLX processed! Total of 243 news
company CRM processed! Total of 242 news
company ORCL processed! Total of 246 news
company NVDA processed! Total of 250 news
company AMD processed! Total of 249 news
company AVGO processed! Total of 246 news
company INTC processed! Total of 245 news
company CSCO processed! Total of 232 news
company IBM processed! Total of 230 news
company TSLA processed! Total of 233 news
company WMT processed! Total of 244 news
company TGT processed! Total of 246 news
company COST processed! Total of 246 news
company MCD processed! Total of 229 news
company SBUX processed! Total of 248 news
company GM processed! Total of 249 news
company HD processed! Total of 236 news
company JPM processed! Total of 245 news
company GS processed! Total of 229 news
comp

{'AAPL': [{'category': 'company',
   'datetime': 1789343597,
   'headline': 'Dow Jones Futures Fall, Techs Tumble As Anthropic Leads Call For AI Slowdown; Fed Meeting Ahead',
   'id': 142126444,
   'image': 'https://s.yimg.com/rz/stage/p/yahoo_finance_en-US_h_p_finance_2.png',
   'related': 'AAPL',
   'source': 'Yahoo',
   'summary': "Anthropic's Dario Amodei, OpenAI's and SpaceX's Elon Musk say they want an AI slowdown. Will they? The Fed looms as well.",
   'url': 'https://finnhub.io/api/news?id=8001ee9da5eb75c60caddb3f42e9733f2df07a6d51cfdccc9bd96c48f74fa851'},
  {'category': 'company',
   'datetime': 1789339800,
   'headline': 'SpaceX vs. Apple: Wall Street Sees Strong Upside for One of These Stocks and Remains Neutral On the Other',
   'id': 142126442,
   'image': 'https://s.yimg.com/rz/stage/p/yahoo_finance_en-US_h_p_finance_2.png',
   'related': 'AAPL',
   'source': 'Yahoo',
   'summary': 'Consensus price targets over the past three months are very bullish on one of these names 

In [ ]:
us_company_codes = [
    # cloud
    "AAPL", "MSFT", "GOOGL", "AMZN", "META", "NFLX", "CRM", "ORCL",
    # chips
    "NVDA", "AMD", "AVGO", "INTC", "CSCO", "IBM",
    # retail
    "TSLA", "WMT", "TGT", "COST", "MCD", "SBUX", "GM", "HD",
    # banking
    "JPM", "GS", "MS", "BAC", "WFC", "AXP", "V", "MA", "BRK.B",
    # sector anchor
    "PG", "KO", "PEP", "JNJ", "PFE", "LLY", "UNH", "GE", "CAT", "BA", "XOM", "CVX"

]

# load news from cache
cached_news_dict = None
with open('news_dict.json', 'r') as f:
    cached_news_dict = json.load(f)

def load_url(url, timeout):
    headers = {
        "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/151.0.0.0 Safari/537.36",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.9",
    }

    response = requests.get(url, headers=headers, timeout=timeout)
    response.raise_for_status()

    return response.content

async def load_url_playwright(url, browser, timeout=30000):
    try:
        page = await browser.new_page()

        await page.goto(url, wait_until="domcontentloaded", timeout=timeout)

        return await page.content()
    finally:
        await page.close()

def get_text_from_html(data):
    soup = BeautifulSoup(data, 'html.parser')
    paragraphs = soup.find_all("p")
    article_text = "\n".join(p.get_text(" ", strip=True).lower() for p in paragraphs)
    return article_text
    
"""
company_articles
{
    'CODE' : {"link":"news article 1", "link": "news article 2", ...}
}
"""
# init empty dict
# company_articles = {code : {} for code in us_company_codes}

# init the cached text
company_articles = None 
with open('news_articles.json', 'r') as f:
    company_articles = json.load(f)

async with async_playwright() as p:
    browser = await p.chromium.launch()

    for code in us_company_codes:
        news_lst = cached_news_dict[code]
        urls = [news['url'] for news in news_lst if news["url"] not in company_articles[code]]

        forbidden_urls = []

        with concurrent.futures.ThreadPoolExecutor(max_workers=8) as executor:
            # Start the load operations and mark each future with its URL
            future_to_url = {executor.submit(load_url, url, 60): url for url in urls}
            for future in concurrent.futures.as_completed(future_to_url):
                url = future_to_url[future]
                try:
                    data = future.result()

                    # convert to text
                    article_text = get_text_from_html(data)

                    # append to list
                    if article_text:
                        company_articles[code][url] = article_text
                        print(f"{url} successfully processed!")

                except Exception as exc:
                    print('%r generated an exception: %s' % (url, exc))
                    if '403' in str(exc):
                        print("forbidden url, adding to forbidden list...")
                        forbidden_urls.append(url)

        if forbidden_urls:
            print(f"There are {len(forbidden_urls)} forbiddin urls, resolving with playwright...")

            for url in forbidden_urls:
                try:
                    data = await load_url_playwright(url, browser)
                    article_text = get_text_from_html(data)

                    if article_text:
                        company_articles[code][url] = article_text
                        print(f"{url} successfully processed via playwright resolution!")

                except Exception as e:
                    print('%r failed with Playwright:  %s' % (url, exc))

        print(f"Company {code} processed! A total of {len(company_articles[code])} out of {len(cached_news_dict[code])} articles processed ")

with open('news_articles.json', 'w') as f:
    json.dump(company_articles , f)



IndentationError: expected an indented block after 'except' statement on line 102 (1449116411.py, line 108)

In [29]:
from playwright.async_api import async_playwright

# async with async_playwright() as p:
#     browser = await p.chromium.launch()
#     page = await browser.new_page()

#     await page.goto('https://finnhub.io/api/news?id=0ab3dc89501a319813963f5ce0faef10bd91fdc5d6fcde435c50ed97d1e895ce')

#     print(await page.title())

#     await browser.close()


async def load_url_playwright(url, timeout=30000):
    async with async_playwright() as p:
        browser = await p.chromium.launch()
        try:
            page = await browser.new_page()

            await page.goto(url, wait_until="domcontentloaded", timeout=timeout)

            return await page.content()
        finally:
            await browser.close()

data = await load_url_playwright('https://finnhub.io/api/news?id=0ab3dc89501a319813963f5ce0faef10bd91fdc5d6fcde435c50ed97d1e895ce')
data

'<!DOCTYPE html><html lang="en"><head><script src="https://pagead2.googlesyndication.com/pagead/managed/js/adsense/m202609100101/show_ads_impl.js" fetchpriority="high"></script><script async="" src="https://scripts.clarity.ms/0.8.69/clarity.js"></script><script type="text/javascript" async="" src="/k4bp/3TC82MvQJhP8ysGZR1enGwB2Qlg58tcqoKLDN4SkqrAjiJyJ"></script><script type="text/javascript" async="" src="https://www.googletagmanager.com/gtag/js?id=GT-WF7TFFD&amp;gtg_health=1"></script><script async="" src="//â€‹cdn.â€‹taboola.â€‹com/libtrc/unip/1969001/tfa.js" id="tb_tfa_script"></script><script async="" src="https://www.clarity.ms/tag/iy8k9ev8b2"></script><script>(function(w,i,g){w[g]=w[g]||[];if(typeof w[g].push==\'function\')w[g].push(i)})\n(window,\'GT-WF7TFFD\',\'google_tags_first_party\');</script><script async="" src="/k4bp/"></script>\n\t\t\t<script>\n\t\t\t\twindow.dataLayer = window.dataLayer || [];\n\t\t\t\tfunction gtag(){dataLayer.push(arguments);}\n\t\t\t\tgtag(\'js\', n

In [ ]:
# a simple strategy using basic functions



## Bigger project planning (paper trade)

Make a bot that can do stuff based on the 5 patterns it sees (three white soldiers, hammer, morning star etc.)